In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report, f1_score

In [2]:
df = pd.read_csv('train.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
# Drop columns that aren't useful for prediction
df = df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])

# Fill missing Age values with the median age
df['Age'] = df['Age'].fillna(df['Age'].median())

# Fill missing Embarked values with the most common value
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Convert text columns into numbers so the model can use them
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
df['Embarked'] = df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,0,22.0,1,0,7.2500,0
1,1,1,1,38.0,1,0,71.2833,1
2,1,3,1,26.0,0,0,7.9250,0
3,1,1,1,35.0,1,0,53.1000,0
4,0,3,0,35.0,0,0,8.0500,0


In [4]:
# X = the features we use to predict, y = what we're trying to predict
X = df.drop(columns=['Survived'])
y = df['Survived']

# Scale features so no single feature (like Fare) dominates due to bigger numbers
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split into 80% training data, 20% testing data
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 712
Testing samples: 179


In [5]:
print("--- Comparing different K values ---")
best_k = None
best_score = 0

for k in [1, 3, 5, 7, 9, 11]:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    score = f1_score(y_test, predictions, average='weighted')
    print(f"K={k:<2}  Weighted F1 Score: {score:.2f}")

    if score > best_score:
        best_score = score
        best_k = k

print(f"\nBest K found: {best_k} (F1 Score: {best_score:.2f})")

--- Comparing different K values ---
K=1   Weighted F1 Score: 0.78
K=3   Weighted F1 Score: 0.80
K=5   Weighted F1 Score: 0.80
K=7   Weighted F1 Score: 0.82
K=9   Weighted F1 Score: 0.80
K=11  Weighted F1 Score: 0.80

Best K found: 7 (F1 Score: 0.82)


In [6]:
# Train the final model using the best K found above
model = KNeighborsClassifier(n_neighbors=best_k)
model.fit(X_train, y_train)
predictions = model.predict(X_test)

print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, predictions))

print("\n--- Classification Report ---")
print(classification_report(y_test, predictions))

final_f1 = f1_score(y_test, predictions, average='weighted')
print(f"Final Weighted F1 Score: {final_f1:.2f}")

--- Confusion Matrix ---
[[92 13]
 [19 55]]

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.83      0.88      0.85       105
           1       0.81      0.74      0.77        74

    accuracy                           0.82       179
   macro avg       0.82      0.81      0.81       179
weighted avg       0.82      0.82      0.82       179

Final Weighted F1 Score: 0.82
